# Notebook to build the script of Circonscription Map generation

## Retrieve Map from INSEE

In [3]:
import requests, geopandas as gpd

url = "https://www.data.gouv.fr/api/1/datasets/r/dbf42fce-a563-412e-9db1-1f7ab453c2ef"
r = requests.get(url)
#read zip file content into a GeoDataFrame
gpf_insee = gpd.read_file(r.content)
#return 5 first rows of the GeoDataFrame
gpf_insee.head(5)

,id_circo,dep,libelle,geometry
0,97302,973,Guyane - 2e circonscription,"MULTIPOLYGON (((-54.60236 2.33356, -54.60242 2..."
1,97301,973,Guyane - 1re circonscription,"MULTIPOLYGON (((-51.93691 4.46648, -51.93667 4..."
2,97201,972,Martinique - 1re circonscription,"MULTIPOLYGON (((-61.00495 14.57791, -61.00492 ..."
3,97202,972,Martinique - 2e circonscription,"MULTIPOLYGON (((-61.12889 14.63162, -61.1289 1..."
4,97203,972,Martinique - 3e circonscription,"POLYGON ((-61.03945 14.64265, -61.0394 14.6425..."


In [4]:
#test the Coordinate Reference System (CRS) of the GeoDataFrame
gpf_insee.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [26]:
gpf_insee.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 558 entries, 0 to 557
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   id_circo  558 non-null    str     
 1   dep       558 non-null    str     
 2   libelle   558 non-null    str     
 3   geometry  558 non-null    geometry
dtypes: geometry(1), str(3)
memory usage: 37.7 KB


### Simplify Map drawing to reduce Map size

1. convert CRS from EPSG:4326 to EPSG:3857 (meter based CRS)

In [5]:
gpf_insee_3857 = gpf_insee.to_crs(epsg=3857)
gpf_insee_3857.head(5)

,id_circo,dep,libelle,geometry
0,97302,973,Guyane - 2e circonscription,"MULTIPOLYGON (((-6078306.99 259842.11, -607831..."
1,97301,973,Guyane - 1re circonscription,"MULTIPOLYGON (((-5781589.848 497711.173, -5781..."
2,97201,972,Martinique - 1re circonscription,"MULTIPOLYGON (((-6791039.757 1640603.641, -679..."
3,97202,972,Martinique - 2e circonscription,"MULTIPOLYGON (((-6804837.228 1646781.86, -6804..."
4,97203,972,Martinique - 3e circonscription,"POLYGON ((-6794879.991 1648050.814, -6794874.6..."


2. In the case of adjacent polygons we use simplify_coverage : warning geometry might need cleaning
```
from shapely import make_valid

# 1. Drop missing/empty and keep only Polygon / MultiPolygon
gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
gdf = gdf[gdf.geom_type.isin(["Polygon", "MultiPolygon"])].copy()

# 2. Repair invalid geometries and remove zero-area artifacts
gdf["geometry"] = gdf.geometry.apply(make_valid)
gdf = gdf[~gdf.geometry.is_empty & (gdf.geometry.area > 0)].copy()

# 3. Simplify coverage with a conservative tolerance first
simplified = gdf.geometry.simplify_coverage(tolerance=0.001)
```

In [51]:
gpf_insee_3857_simplified = gpf_insee_3857[gpf_insee_3857.geometry.notna()].copy()

In [54]:
gpf_insee_3857_simplified.geometry.is_valid_coverage()

False

Follow the process : https://www.geospatialdatatesting.org/spatial-test-pattern-design-implementation/geometry-validation-patterns/repairing-invalid-geometries-with-make-valid-safely/

In [55]:
gpf_insee_3857_simplified.geometry.is_valid_reason()

0                                         Valid Geometry
1                                         Valid Geometry
2                                         Valid Geometry
3                                         Valid Geometry
4                                         Valid Geometry
                             ...                        
553                                       Valid Geometry
554    Self-intersection[747701.925307082 5483968.441...
555                                       Valid Geometry
556                                       Valid Geometry
557                                       Valid Geometry
Length: 558, dtype: str

In [56]:
gpf_insee_3857_simplified.geometry.is_valid_reason()[gpf_insee_3857_simplified.geometry.is_valid_reason() != "Valid Geometry"]

19     Self-intersection[634672.520235714 5651160.983...
22     Self-intersection[668147.618139357 5654205.621...
24     Self-intersection[156139.667014223 5378526.600...
36     Self-intersection[-71034.5301356465 5599665.65...
42     Self-intersection[75309.6242819772 6107546.903...
                             ...                        
544    Self-intersection[-88878.3557728396 6115872.19...
546    Ring Self-intersection[-320849.461092849 60668...
548    Self-intersection[-456021.289501545 6097293.27...
549    Self-intersection[-485002.987010252 6193042.35...
554    Self-intersection[747701.925307082 5483968.441...
Length: 71, dtype: str

In [57]:
gpf_insee_3857_simplified.geometry[gpf_insee_3857_simplified.geometry.is_valid_reason() != "Valid Geometry"].make_valid()

19     MULTIPOLYGON (((634481.924 5646816.987, 634481...
22     MULTIPOLYGON (((613169.755 5632562.031, 613169...
24     GEOMETRYCOLLECTION (MULTIPOLYGON (((156139.667...
36     MULTIPOLYGON (((-71059.251 5599613.032, -71059...
42     MULTIPOLYGON (((-11473.002 6061609.261, -11473...
                             ...                        
544    MULTIPOLYGON (((-72779.159 6062613.078, -72779...
546    GEOMETRYCOLLECTION (POLYGON ((-321574.574 6059...
548    MULTIPOLYGON (((-457416.532 6097335.562, -4560...
549    MULTIPOLYGON (((-474065.614 6169029.649, -4741...
554    MULTIPOLYGON (((728198.772 5430802.255, 728573...
Length: 71, dtype: geometry

In [28]:
gpf_insee_3857_simplified = gpf_insee_3857_simplified[gpf_insee_3857_simplified.geometry.type.isin(["Polygon","MultiPolygon"])].copy()

In [29]:
from shapely.validation import make_valid
#gpf_insee_3857_simplified = gpf_insee_3857_simplified.geometry.apply(make_valid)
gpf_insee_3857_simplified = gpf_insee_3857_simplified[gpf_insee_3857_simplified.geometry.notna() & gpf_insee_3857_simplified.geometry.area > 0].copy()

In [30]:
set(gpf_insee_3857_simplified.geometry.type)

{'MultiPolygon', 'Polygon'}

In [44]:
import shapely
list_num = []
for i in range(len(gpf_insee_3857_simplified)):
    list_num.append(shapely.get_num_coordinates(gpf_insee_3857_simplified.geometry[i]))

In [48]:
for i in range(len(list_num)):
    if list_num[i] == min(list_num):
        print(i)

134


In [50]:
gpf_insee_3857_simplified.iloc[134]

id_circo                                                 9209
dep                                                        92
libelle                   Hauts-de-Seine - 9e circonscription
geometry    POLYGON ((248064.03288606368 6245873.309682483...
Name: 134, dtype: object

In [32]:
gpf_insee_3857_simplified.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 558 entries, 0 to 557
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   id_circo  558 non-null    str     
 1   dep       558 non-null    str     
 2   libelle   558 non-null    str     
 3   geometry  558 non-null    geometry
dtypes: geometry(1), str(3)
memory usage: 37.7 KB


In [33]:
gpf_insee_3857_simplified = gpf_insee_3857_simplified.simplify_coverage(tolerance=0.001)

GEOSException: IllegalArgumentException: point array must contain 0 or >1 elements


### Tests few cases at circonscription border

## Retrieve COM (Communes D'Outre Mer) from ArcGIS Science PO service

### Download map of COM from Science PO service and change CRS (Coordinate Reference System)

## Concat Maps

## Export File to designated folder